## Model selection:


In [ ]:
!pip install xgboost

: 

In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.keras

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from tensorflow.keras.models import load_model
import numpy as np


def load_and_prepare_data():
    data = pd.read_pickle("database/feature_engineered.pkl")
    target_column = 'risk_flag'
    X = data.drop(columns=[target_column, 'applicant_id'])

    categorical_cols = ['married_single', 'house_ownership', 'car_ownership', 'city', 'state']
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    X.columns = X.columns.str.replace(r"[\[\]<>]", "", regex=True)

    y = data[target_column]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.66, random_state=42, stratify=y_temp
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def train_and_evaluate_model(model, X_train, y_train, X_val, y_val, model_name):
    print(f"\nTraining and evaluating {model_name}...")

    with mlflow.start_run(run_name=model_name):
        if model_name == "Neural Network":
            y_pred_probs = model.predict(X_val).flatten()
            y_pred = (y_pred_probs > 0.5).astype(int)
            acc = accuracy_score(y_val, y_pred)

            mlflow.log_metric("accuracy", acc)
            mlflow.keras.log_model(model, artifact_path="nn_model")
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            acc = accuracy_score(y_val, y_pred)

            mlflow.log_metric("accuracy", acc)
            mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"{model_name} Validation Accuracy: {acc:.4f}")
        print(f"{model_name} Classification Report:\n{classification_report(y_val, y_pred)}")


def main():
    X_train, X_val, X_test, y_train, y_val, y_test = load_and_prepare_data()

    nn_model = load_model("trained_models/nn_model.h5")

    models = {
        "Logistic Regression": LogisticRegression(max_iter=5000, class_weight='balanced'),
        "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
        "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "Neural Network": nn_model
    }

    mlflow.set_experiment("Phase3_Model_Comparison")

    for name, model in models.items():
        train_and_evaluate_model(model, X_train, y_train, X_val, y_val, name)


if __name__ == "__main__":
    main()


## Find Param : 

In [7]:
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.keras

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.models import load_model
import joblib

def load_and_prepare_data():
    data = pd.read_pickle("database/feature_engineered.pkl")
    target_column = 'risk_flag'
    X = data.drop(columns=[target_column, 'applicant_id'])

    categorical_cols = ['married_single', 'house_ownership', 'car_ownership', 'city', 'state']
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    X.columns = X.columns.str.replace(r"[\[\]<>]", "", regex=True)

    y = data[target_column]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.66, random_state=42, stratify=y_temp
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

def evaluate_nn_model(X_val, y_val, X_test, y_test):
    print("\nEvaluating Neural Network model...")
    model = load_model("trained_models/nn_model.h5")

    y_val_pred = (model.predict(X_val) > 0.5).astype(int).flatten()
    y_test_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    print("Validation Accuracy (NN):", val_acc)
    print("Validation Classification Report:\n", classification_report(y_val, y_val_pred))

    print("Test Accuracy (NN):", test_acc)
    print("Test Classification Report:\n", classification_report(y_test, y_test_pred))

    mlflow.log_metric("NN_val_accuracy", val_acc)
    mlflow.log_metric("NN_test_accuracy", test_acc)

def main():
    mlflow.set_experiment("Phase3_RandomForest_GridSearch")

    X_train, X_val, X_test, y_train, y_val, y_test = load_and_prepare_data()

    print("Starting Grid Search for hyperparameter tuning...")

    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, None],
        #'min_samples_split': [2, 5],
        #'min_samples_leaf': [1, 2],
        'class_weight': ['balanced']
    }

    rf = RandomForestClassifier(random_state=42)

    with mlflow.start_run(run_name="RandomForest_GridSearch") as run:
        grid_search = GridSearchCV(
            estimator=rf,
            param_grid=param_grid,
            cv=3,
            scoring='f1',
            n_jobs=-1,
            verbose=2
        )

        grid_search.fit(X_train, y_train)

        print(f"Best parameters found: {grid_search.best_params_}")
        mlflow.log_params(grid_search.best_params_)

        best_rf = grid_search.best_estimator_

        print("Evaluating best model on validation set:")
        y_val_pred = best_rf.predict(X_val)
        val_acc = accuracy_score(y_val, y_val_pred)
        print(f"Validation Accuracy: {val_acc:.4f}")
        print(classification_report(y_val, y_val_pred))

        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.sklearn.log_model(best_rf, artifact_path="random_forest_best_model")

        print("Training final model on combined training and validation data...")
        X_final_train = pd.concat([X_train, X_val])
        y_final_train = pd.concat([y_train, y_val])
        best_rf.fit(X_final_train, y_final_train)

        print("Evaluating final model on test set:")
        y_test_pred = best_rf.predict(X_test)
        test_acc = accuracy_score(y_test, y_test_pred)
        print(f"Test Accuracy: {test_acc:.4f}")
        print(classification_report(y_test, y_test_pred))

        mlflow.log_metric("test_accuracy", test_acc)

        joblib.dump(best_rf, "trained_models/random_forest_final.pkl")
        print("Model saved to trained_models/random_forest_final.pkl")

        evaluate_nn_model(X_val, y_val, X_test, y_test)

if __name__ == "__main__":
    main()


Starting Grid Search for hyperparameter tuning...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END class_weight=balanced, max_depth=10, n_estimators=100; total time=  37.9s
[CV] END class_weight=balanced, max_depth=10, n_estimators=100; total time=  47.6s
[CV] END class_weight=balanced, max_depth=10, n_estimators=100; total time=  48.8s
[CV] END class_weight=balanced, max_depth=10, n_estimators=200; total time= 1.1min
[CV] END class_weight=balanced, max_depth=10, n_estimators=200; total time= 1.2min
[CV] END class_weight=balanced, max_depth=10, n_estimators=200; total time= 1.3min
[CV] END class_weight=balanced, max_depth=None, n_estimators=100; total time= 3.2min
[CV] END class_weight=balanced, max_depth=None, n_estimators=100; total time= 3.3min
[CV] END class_weight=balanced, max_depth=None, n_estimators=100; total time= 3.3min
Best parameters found: {'class_weight': 'balanced', 'max_depth': None, 'n_estimators': 200}
Evaluating best model on validation set:
Vali

2025/06/06 04:47:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Training final model on combined training and validation data...
[CV] END class_weight=balanced, max_depth=None, n_estimators=200; total time= 4.9min
[CV] END class_weight=balanced, max_depth=None, n_estimators=200; total time= 5.0min
[CV] END class_weight=balanced, max_depth=None, n_estimators=200; total time= 5.0min
Evaluating final model on test set:
Test Accuracy: 0.8952
              precision    recall  f1-score   support

           0       0.97      0.91      0.94     43759
           1       0.55      0.77      0.64      6137

    accuracy                           0.90     49896
   macro avg       0.76      0.84      0.79     49896
weighted avg       0.92      0.90      0.90     49896

Model saved to trained_models/random_forest_final.pkl

Evaluating Neural Network model...


804/804 ━━━━━━━━━━━━━━━━━━━━ 1s 920us/step
1560/1560 ━━━━━━━━━━━━━━━━━━━━ 1s 814us/step
Validation Accuracy (NN): 0.8959305944600062
Validation Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.97      0.94     22542
           1       0.64      0.36      0.46      3162

    accuracy                           0.90     25704
   macro avg       0.78      0.67      0.70     25704
weighted avg       0.88      0.90      0.88     25704

Test Accuracy (NN): 0.895422478755812
Test Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.97      0.94     43759
           1       0.63      0.37      0.47      6137

    accuracy                           0.90     49896
   macro avg       0.77      0.67      0.70     49896
weighted avg       0.88      0.90      0.88     49896

